# Easy Italian News
- download and clean up episodes
- download mp3 files as well
- this is good for 1 month (can change date to do it for other months)

# Find all days with news

In [ ]:
import requests
from bs4 import BeautifulSoup
import re

l_to_remove = [#r'\n+', 
    r'www\..*\n',
    r'Click the link..*\n', 
    r'https://..*\n', 
    r'\xa0\n', 
    r'Creative Commons..*\n',
    r'Image.*\n', 
    # r'it\.[\w+]..*\n', 'tg24\..*\n',
    # r'[\s]Jeffrey Zeldman\n',
    # r'[\s]Zdravko Petrov\n', 
    # r'[\s]Chris Watt\n',
    # r'[\s]GovernmentZA.*\n',
    # r'[\s]Attribution..*\n',
    # r'[\s]Elvert Barnes..*\n',
    # r'[\s]Maritza Ríos..*\n', 
    # r'[\s]si.robi.*\n',
    # r'[\s]Steven Depolo.*\n',
    # r'[\s]Brendan Keene.*\n',
    # r'[\s]Francesco Ranieri.*\n',
    # r'[\s]Nathan Keirn.*\n',
    # r'[\s]Giorgio Minguzzi.*\n',
    #r'^[\w+]\.[\w+]\.[\w+]$'
    #r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    
    r' \n',    
]

def merge_paragraphs(text):
    # Split on one or more empty lines
    blocks = re.split(r'\n\s*\n+', text.strip())

    paragraphs = []
    for block in blocks:
        # Remove leading/trailing whitespace from each line
        lines = [line.strip() for line in block.splitlines() if line.strip()]

        # Join lines within the block into a single paragraph
        paragraph = " ".join(lines)

        if paragraph:
            paragraphs.append(paragraph)

    return paragraphs

# Pattern to remove 2-4 words separated by dots.
# The only space can be at the beginnign of the line
pattern = re.compile(
    r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    re.MULTILINE
)

In [ ]:
# Specify the URL of the website you want to scrape
url = 'https://easyitaliannews.com/2026/04/'

# To avoid server error: 403
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/117.0"
}

# Send a GET request to the URL
response = requests.get(url, headers=headers)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    l_tds = soup.find_all('td')

    l_of_news = []
    for td in soup.find_all('td'):
        try:
            l_of_news.append(td.find('a').get('href'))
        except:
            pass

In [ ]:
l_of_news

In [ ]:
pattern = re.compile(
    r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    re.MULTILINE
)

In [ ]:
for url in l_of_news:
#for url in l_of_news[:1]:
    news_date = url[28:38].replace('/', '-')
    print(news_date)

    # Send a GET request to the URL
    response = requests.get(url, headers=headers)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')

        for h4 in soup.find_all("h4"):
            p = soup.new_tag("p")
            p.string = "## " + h4.get_text(strip=True)
            #p.append(p)
        
            h4.replace_with(p)
            
            # Add two line breaks after the paragraph
            p.insert_after(soup.new_tag("br"))
            p.insert_after(soup.new_tag("br"))
            

        ns = soup.find('div', {'class': 'entry-content'})

        # remove <strong> tags
        for strong in ns.find_all("strong"):
            strong.unwrap()

        # Extracting all paragraphs
        l_content = []
        mp3 = ''
        paragraphs = ns.find_all(True)
        for idx, paragraph in enumerate(paragraphs):
            # print(f"Paragraph {idx+1}: {paragraph.text}")
            p = str(paragraph.text)
        #     if  p not in l_content:
        #         l_content.append(p)
            if mp3 == '' and 'mp3' in p:
                mp3 = p.strip()
            l_content.append(p)

        s = ('\n').join(l_content).split('Il tuo aiuto per noi è importante!')[0].split('Subscribe')[0]


        t = ''
        for el in l_to_remove:
            t = re.sub(el, '\n', s)
            s = t

        # Remove the links to websites (2-4 words separate by dots)
        cleaned_text = pattern.sub('', s)

        # Group text by paragraphs
        paragraphed = merge_paragraphs(cleaned_text)

        # Output with blank line before paragraphs that start with a capitalized word
        result = []
        for i, para in enumerate(paragraphed):
            # print(para)
            # if i > 0 and re.match(r"^[A-Z]*[^\s]*$", para.split()[0]):
            #     result.append("")  # blank line
            if re.fullmatch(r'[A-Z]+', para.split()[0]):
                result.append("")  # blank line
            result.append(para)
        
        final_text = "\n".join(result)
        
        # # Remove duplicated lines!
        # s = re.sub(r'^(.*)(\r?\n)\1(\r?\n)?', r'\1\2', text, flags=re.MULTILINE)

        
        # # Replace 2 or more consecutive blank lines with a single blank line
        # text = re.sub(r'\n\s*\n+', '\n\n', cleaned_text)

        
        with open(f'EasyItalianNews_{news_date}.txt', 'w', encoding='utf-8') as f:
            f.write(final_text)

        # doc = requests.get(mp3)

        # with open(f'EasyItalianNews_{news_date}.mp3', 'wb') as f:
        #     f.write(doc.content)
print('done')